In [60]:
import math
import pandas as pd
import numpy as np

In [61]:
def normal_cdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    return (1 + math.erf((x - mu) / (sigma * math.sqrt(2)))) / 2

In [92]:
def mann_whitney_wilcoxons_t_test():
    data = []
    with open("Uptime of electrical equipment in the HOA building.csv", "r", encoding = "utf-8-sig") as f:
        header_line = f.readline()
        for line in f:
            parts = line.strip().split(";")
            if len(parts) == 3:
                data.append([int(parts[0]), int(parts[1]), float(parts[2])])
    df = pd.DataFrame(data, columns = ["House Number", "Belongs to a Homeowners' Association", "Uninterrupted Operating Time (hours)"])
    sorted_df = df.sort_values(by = "Uninterrupted Operating Time (hours)", ascending = True).reset_index(drop = True)
    sorted_df["Overall Rank (1 - 500)"] = range(1, (len(sorted_df) + 1))
    ranks_sample_num_1 = sorted_df[sorted_df["Belongs to a Homeowners' Association"] == 1]["Overall Rank (1 - 500)"].reset_index(drop = True)
    ranks_sample_num_2 = sorted_df[sorted_df["Belongs to a Homeowners' Association"] == 0]["Overall Rank (1 - 500)"].reset_index(drop = True)
    sorted_df["Ranks of the elements in the first sample"] = ranks_sample_num_1
    sorted_df["Ranks of the elements in the second sample"] = ranks_sample_num_2
    W_with_star_1 = (
        (2 * ranks_sample_num_1.sum()
         - len(ranks_sample_num_1) * (len(ranks_sample_num_1) + len(ranks_sample_num_2) + 1)
         + 1)
        /
        np.sqrt(
            (len(ranks_sample_num_1) * len(ranks_sample_num_2) / 3)
            * (len(ranks_sample_num_1) + len(ranks_sample_num_2) + 1)
        )
    )
    W_with_star_2 = (
        (2 * ranks_sample_num_2.sum() 
         - len(ranks_sample_num_2) * (len(ranks_sample_num_1) + len(ranks_sample_num_2) + 1) + 1) 
        / 
        np.sqrt(
            (len(ranks_sample_num_1) * len(ranks_sample_num_2) / 3) 
            * (len(ranks_sample_num_1) + len(ranks_sample_num_2) + 1)
        )
    )
    p_value_1 = 2 * (1 - normal_cdf(abs(W_with_star_1)))
    p_value_2 = 2 * (1 - normal_cdf(abs(W_with_star_2)))
    print(f"Wilcoxon's criterion W* (Sample # 1): {W_with_star_1:.3f}")
    print(f"p-value (Sample # 1): {p_value_1:.4f}\n")
    print(f"Wilcoxon's criterion W* (Sample #2): {W_with_star_2:.3f}")
    print(f"p-value (Sample # 2): {p_value_2:.4f}\n")
    display(sorted_df.head(10)) 
    sorted_df.to_csv("Ranks Separated uptime of electrical equipment in the HOA building.csv", index = False, encoding = "utf-8-sig")

In [93]:
mann_whitney_wilcoxons_t_test()

Wilcoxon's criterion W* (Sample # 1): 1.093
p-value (Sample # 1): 0.2742

Wilcoxon's criterion W* (Sample #2): -1.093
p-value (Sample # 2): 0.2745



,House Number,Belongs to a Homeowners' Association,Uninterrupted Operating Time (hours),Overall Rank (1 - 500),Ranks of the elements in the first sample,Ranks of the elements in the second sample
0,200,1,85.0,1,1.0,2.0
1,454,0,85.0,2,3.0,5.0
2,196,1,85.0,3,4.0,7.0
3,303,1,85.0,4,6.0,8.0
4,298,0,86.0,5,11.0,9.0
5,346,1,86.0,6,12.0,10.0
6,321,0,86.0,7,14.0,13.0
7,16,0,86.0,8,19.0,15.0
8,487,0,86.0,9,22.0,16.0
9,56,0,87.0,10,23.0,17.0


In [112]:
def siegel_tukey_test():
    df = pd.read_csv("Uptime of Laptops from Various Brands.csv", sep = ";", encoding = "utf-8-sig")
    
    df.columns = ["Laptops Serial Number", "Belongs to a Computer Producers Brand", "Uninterrupted Operating Time (hours)"]         
    sorted_df = df.sort_values(by = "Uninterrupted Operating Time (hours)", ascending = True).reset_index(drop = True)
    
    n_total = len(sorted_df)
    siegel_ranks = [0] * n_total
    left = 0
    right = n_total - 1
    current_rank = 1
    
    while left <= right:
        if left <= right:
            siegel_ranks[left] = current_rank
            current_rank += 1
            left += 1
        if left <= right:
            siegel_ranks[right] = current_rank
            current_rank += 1
            right -= 1
        if left <= right:
            siegel_ranks[right] = current_rank
            current_rank += 1
            right -= 1
        if left <= right:
            siegel_ranks[left] = current_rank
            current_rank += 1
            left += 1
            
    sorted_df["Siegel-Tukey Rank"] = siegel_ranks
    ranks_sample_num_1 = sorted_df[sorted_df["Belongs to a Computer Producers Brand"] == 1]["Siegel-Tukey Rank"].reset_index(drop = True)
    ranks_sample_num_2 = sorted_df[sorted_df["Belongs to a Computer Producers Brand"] == 0]["Siegel-Tukey Rank"].reset_index(drop = True)
    
    Z = (
        (2 * ranks_sample_num_1.sum() 
         - len(ranks_sample_num_1) * (len(ranks_sample_num_1) + len(ranks_sample_num_2) + 1) 
         + 1) 
        / 
        np.sqrt(
            (len(ranks_sample_num_1) * len(ranks_sample_num_2) / 3) 
            * (len(ranks_sample_num_1) + len(ranks_sample_num_2) + 1)
        )
    )
    p_value = 2 * (1 - normal_cdf(abs(Z)))
    
    print(f"Siegel-Tukey's Test Z: {Z:.4f}")
    print(f"p-value: {p_value:.4f}\n")
    
    display(sorted_df.head(10)) 
    sorted_df.to_csv("Ranks Separated Uptime of Laptops from Various Brands.csv", index=False, encoding="utf-8-sig")

In [113]:
siegel_tukey_test()

Siegel-Tukey's Test Z: 0.9992
p-value: 0.3177



,Laptops Serial Number,Belongs to a Computer Producers Brand,Uninterrupted Operating Time (hours),Siegel-Tukey Rank
0,1614807,1,85,1
1,1207197,0,86,4
2,1486755,0,87,5
3,1118982,1,87,8
4,1876689,1,87,9
5,1389432,0,88,12
6,1293261,1,88,13
7,1199894,0,88,16
8,1869211,0,88,17
9,1077939,0,89,20
